# Fake Reviews Dataset - Fusion Model

Combines TF-IDF (raw text patterns) with engineered linguistic features (readability, lexical diversity, sentiment, POS ratios) into a single feature matrix, then trains classifiers on top.

**Rationale:** the baseline notebook showed:
- TF-IDF alone: ~88% accuracy (captures vocabulary/phrasing)
- Engineered features alone: ~79-83% accuracy (captures structural/style patterns)

## Setup

In [1]:
import numpy as np
import pandas as pd
from scipy.sparse import hstack, csr_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

RANDOM_STATE = 42

ENGINEERED_COLS = [
    "word_count", "char_count", "avg_word_length", "sentence_count",
    "avg_sentence_length", "flesch_reading_ease", "flesch_kincaid_grade",
    "unique_word_ratio", "exclamation_ratio", "capital_word_ratio",
    "punctuation_density", "sentiment_polarity", "sentiment_subjectivity",
    "adjective_ratio", "noun_ratio", "verb_ratio",
]


## Configuration

In [2]:
INPUT_PATH = "data/processed/fake_reviews_merged.csv"  # must contain review_text + engineered features + label_clean
MAX_TFIDF_FEATURES = 5000


## Evaluation Helper

In [3]:
def evaluate(name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, pos_label="fake")
    rec = recall_score(y_true, y_pred, pos_label="fake")
    f1 = f1_score(y_true, y_pred, pos_label="fake")
    cm = confusion_matrix(y_true, y_pred, labels=["real", "fake"])

    print(f"\n{'='*60}")
    print(f"  {name}")
    print(f"{'='*60}")
    print(f"Accuracy:            {acc:.4f}")
    print(f"Precision (fake):    {prec:.4f}")
    print(f"Recall (fake):       {rec:.4f}")
    print(f"F1 (fake):           {f1:.4f}")
    print("Confusion matrix [rows=actual, cols=predicted] order=[real, fake]:")
    print(cm)
    print(classification_report(y_true, y_pred, digits=4))

    return {"model": name, "accuracy": acc, "precision_fake": prec,
            "recall_fake": rec, "f1_fake": f1}


## 1. Load & Validate

In [4]:
df = pd.read_csv(INPUT_PATH)

required = ["review_text", "label_clean"] + ENGINEERED_COLS
missing = [c for c in required if c not in df.columns]
df = df.dropna(subset=required).reset_index(drop=True)
print(f"Rows after dropping missing values: {len(df)}")
df.head()

Rows after dropping missing values: 40403


,review_text,word_count,char_count,avg_word_length,sentence_count,avg_sentence_length,flesch_reading_ease,flesch_kincaid_grade,unique_word_ratio,exclamation_ratio,capital_word_ratio,punctuation_density,sentiment_polarity,sentiment_subjectivity,adjective_ratio,noun_ratio,verb_ratio,rating,category,label_clean
0,"Love this! Well made, sturdy, and very comfort...",12,73,5.166667,4,3.0,73.845000,4.450000,0.916667,16.666667,0.0,0.068493,0.473750,0.8,0.111111,0.055556,0.166667,5.0,Home_and_Kitchen_5,fake
1,"love it, a great upgrade from the original. I'...",16,79,4.000000,2,8.0,87.677500,3.017500,0.937500,0.000000,0.0,0.037975,0.558333,0.7,0.105263,0.210526,0.157895,5.0,Home_and_Kitchen_5,fake
2,This pillow saved my back. I love the look and...,14,67,3.857143,2,7.0,103.044286,0.625714,0.928571,0.000000,0.0,0.029851,0.250000,0.3,0.000000,0.312500,0.125000,5.0,Home_and_Kitchen_5,fake
3,"Missing information on how to use it, but it i...",17,80,3.764706,2,8.5,80.097647,6.310588,1.000000,5.882353,0.0,0.025000,0.400000,0.4,0.052632,0.157895,0.157895,1.0,Home_and_Kitchen_5,fake
4,Very nice set. Good quality. We have had the s...,18,85,3.777778,3,6.0,99.000000,1.686667,0.944444,0.000000,0.0,0.023529,0.740000,0.8,0.100000,0.150000,0.250000,5.0,Home_and_Kitchen_5,fake


## 2. Train/Test Split

Split indices first, so text and engineered features stay aligned.

In [5]:
y = df["label_clean"].values
idx_train, idx_test = train_test_split(np.arange(len(df)), test_size=0.2, random_state=RANDOM_STATE, stratify=y)


## 3. TF-IDF Part

In [6]:
vectorizer = TfidfVectorizer(
    max_features=MAX_TFIDF_FEATURES, ngram_range=(1, 2), min_df=2, stop_words="english"
)
X_tfidf_train = vectorizer.fit_transform(df["review_text"].iloc[idx_train].astype(str))
X_tfidf_test = vectorizer.transform(df["review_text"].iloc[idx_test].astype(str))


## 4. Engineered Features Part

Scaled, then converted to sparse to combine with TF-IDF.

In [7]:
scaler = StandardScaler()
X_eng_train = scaler.fit_transform(df[ENGINEERED_COLS].iloc[idx_train])
X_eng_test = scaler.transform(df[ENGINEERED_COLS].iloc[idx_test])

## 5. Fuse: Horizontally Stack TF-IDF + Engineered Features

In [8]:
X_train = hstack([X_tfidf_train, csr_matrix(X_eng_train)]).tocsr()
X_test = hstack([X_tfidf_test, csr_matrix(X_eng_test)]).tocsr()

y_train, y_test = y[idx_train], y[idx_test]

print(f"\nFused feature matrix shape: train={X_train.shape}, test={X_test.shape}")
print(f"  (TF-IDF dims: {X_tfidf_train.shape[1]}, engineered dims: {len(ENGINEERED_COLS)})")



Fused feature matrix shape: train=(32322, 5016), test=(8081, 5016)
  (TF-IDF dims: 5000, engineered dims: 16)


## 6. Train & Evaluate Models

In [9]:
results = []

lr = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
lr.fit(X_train, y_train)
results.append(evaluate("Fusion (TF-IDF + Engineered) - Logistic Regression", y_test, lr.predict(X_test)))

svm = LinearSVC(random_state=RANDOM_STATE)
svm.fit(X_train, y_train)
results.append(evaluate("Fusion (TF-IDF + Engineered) - Linear SVM", y_test, svm.predict(X_test)))



  Fusion (TF-IDF + Engineered) - Logistic Regression
Accuracy:            0.8973
Precision (fake):    0.9046
Recall (fake):       0.8881
F1 (fake):           0.8963
Confusion matrix [rows=actual, cols=predicted] order=[real, fake]:
[[3665  378]
 [ 452 3586]]
              precision    recall  f1-score   support

        fake     0.9046    0.8881    0.8963      4038
        real     0.8902    0.9065    0.8983      4043

    accuracy                         0.8973      8081
   macro avg     0.8974    0.8973    0.8973      8081
weighted avg     0.8974    0.8973    0.8973      8081


  Fusion (TF-IDF + Engineered) - Linear SVM
Accuracy:            0.9036
Precision (fake):    0.9089
Recall (fake):       0.8970
F1 (fake):           0.9029
Confusion matrix [rows=actual, cols=predicted] order=[real, fake]:
[[3680  363]
 [ 416 3622]]
              precision    recall  f1-score   support

        fake     0.9089    0.8970    0.9029      4038
        real     0.8984    0.9102    0.9043      4043

## Summary

In [10]:
summary = pd.DataFrame(results)
summary

,model,accuracy,precision_fake,recall_fake,f1_fake
0,Fusion (TF-IDF + Engineered) - Logistic Regres...,0.897290,0.904642,0.888063,0.896276
1,Fusion (TF-IDF + Engineered) - Linear SVM,0.903601,0.908908,0.896979,0.902904
